## First introduction to data, used as a tool to help us understand the data and the problem

In [60]:
import pandas as pd
import numpy as np
import seaborn as sns

## Import data

In [61]:
cross_border_payments = pd.read_csv("../data/cross_border_payments.csv")
trade_finance = pd.read_csv("../data/trade_finance.csv")
transactional_banking = pd.read_csv("../data/transactional_banking.csv")

## Exploring Cross-Border Payments Data

Before building any wallet-sizing logic on this dataset, we explore its structure and 
check whether each field carries real signal or is closer to noise. This determines 
how we treat `corridor_type`, `currency_pair`, and `memo` in the wallet estimation model.

In [62]:
cross_border_payments.head()

,transaction_id,entity_id,entity_name,sector,date,direction,currency_pair,value_zar,counterparty_country,corridor_type,beneficiary_name,reference,memo
0,XBP63220455,E01,BHP Group,mining,2023-07-01,inbound,USD/ZAR,2541553.55,Switzerland,intercompany,BHP Group Switzerland Ltd,INTERCO-730855,NaN
1,XBP14207725,E11,Pepkor Holdings,consumer,2023-07-01,outbound,USD/ZAR,407839.87,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-827118,NaN
2,XBP66460952,E11,Pepkor Holdings,consumer,2023-07-01,outbound,CNY/ZAR,72148.47,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-488519,NaN
3,XBP45973312,E11,Pepkor Holdings,consumer,2023-07-01,outbound,GBP/ZAR,53285.65,Namibia,intercompany,Pepkor Holdings Namibia Ltd,INTERCO-585129,NaN
4,XBP13173829,E11,Pepkor Holdings,consumer,2023-07-01,inbound,AED/ZAR,2858193.91,Japan,trade,Continental Resources Trading,TRADE-568825,NaN


### Initial exploration

Checking: how `corridor_type` splits by count and value, whether `memo` is populated, 
currency pair distribution, and how activity is spread across clients.

In [76]:
print("Corridor type counts:")
print(cross_border_payments['corridor_type'].value_counts())

print("\nTotal ZAR value by corridor type:")
print(cross_border_payments.groupby('corridor_type')['value_zar'].sum())

print("\nMemo populated rows:", cross_border_payments['memo'].notna().sum(),
      "of", len(cross_border_payments))

print("\nCurrency pair counts:")
print(cross_border_payments['currency_pair'].value_counts())

print("\nTransactions per client (entity_id):")
print(cross_border_payments.groupby('entity_id')['transaction_id'].count().describe())

Corridor type counts:
corridor_type
trade           109187
intercompany    107969
other            23961
Name: count, dtype: int64

Total ZAR value by corridor type:
corridor_type
intercompany    6.015801e+10
other           1.326243e+10
trade           6.032625e+10
Name: value_zar, dtype: float64

Memo populated rows: 448 of 241117

Currency pair counts:
currency_pair
EUR/ZAR    48339
GBP/ZAR    48285
USD/ZAR    48243
AED/ZAR    48189
CNY/ZAR    48061
Name: count, dtype: int64

Transactions per client (entity_id):
count       20.000000
mean     12055.850000
std      10630.497029
min       1109.000000
25%       4983.500000
50%       8760.500000
75%      15917.000000
max      43043.000000
Name: transaction_id, dtype: float64


### Data scope note

The three datasets provided (cross-border payments, transactional banking, trade 
finance) contain records for **20 clients (E01–E20) only**. Although the brief 
describes a 50-client portfolio, no transactional or reference data — and no client 
master file — was provided for the remaining 30 clients (E21–E50).

This analysis is scoped to the 20 clients present in the data. This is a data-scope 
limitation, not a wallet-share finding — we make no claims about E21–E50 since we have 
no data on them at all.

`trade` and `intercompany` are near-identical in count and value, with almost the same 
average transaction size — no clear size-based distinction between them. Since 
intercompany flows move money between a company and its own subsidiaries, they're not 
genuinely contestable banking business, so we exclude them from wallet estimation. 
`currency_pair` is also close to evenly distributed across all 5 currencies - treated 
as a weak signal, not used for sizing. `trade` and `other` remain to investigate.

### Investigating `memo` — sparse but informative where present

Only 448 of 241,117 rows (0.2%) have `memo` populated. Checking which corridor type 
these cluster in, and what the memo text actually says.

In [64]:
memo_rows = cross_border_payments[cross_border_payments['memo'].notna()]

print("Corridor type of memo-populated rows:")
print(memo_rows['corridor_type'].value_counts())

print("\nMemo phrase breakdown:")
print(memo_rows['memo'].str.split(' - ref| - ext\\. ref', regex=True).str[0].value_counts())

Corridor type of memo-populated rows:
corridor_type
trade    448
Name: count, dtype: int64

Memo phrase breakdown:
memo
Bridging facility settlement          130
Syndicate participation settlement    127
Settlement re: facility drawdown      109
Loan drawdown proceeds                 82
Name: count, dtype: int64


### Investigating `other` — is it a meaningful category or a residual bucket?

`other` has no memo overlap, so it isn't the same lending signal. Checking whether it 
differs from `trade`/`intercompany` on direction, country, or client distribution.

In [65]:
other_rows = cross_border_payments[cross_border_payments['corridor_type'] == 'other']

print("'other' — direction split:")
print(other_rows['direction'].value_counts())

print("\n'other' — top counterparty countries:")
print(other_rows['counterparty_country'].value_counts().head(10))

print("\n'other' — transactions per client:")
print(other_rows.groupby('entity_id').size().describe())

print("\nDirection split, all corridor types (comparison):")
print(cross_border_payments.groupby('corridor_type')['direction'].value_counts())

'other' — direction split:
direction
outbound    12460
inbound     11501
Name: count, dtype: int64

'other' — top counterparty countries:
counterparty_country
Germany                 1925
Brazil                  1924
India                   1921
Japan                   1898
Netherlands             1881
United Kingdom          1874
United Arab Emirates    1859
Switzerland             1859
United States           1843
China                   1842
Name: count, dtype: int64

'other' — transactions per client:
count      20.00000
mean     1198.05000
std      1053.89925
min       137.00000
25%       498.00000
50%       893.50000
75%      1520.75000
max      4229.00000
dtype: float64

Direction split, all corridor types (comparison):
corridor_type  direction
intercompany   outbound     56393
               inbound      51576
other          outbound     12460
               inbound      11501
trade          outbound     56941
               inbound      52246
Name: count, dtype: int64


### Key Findings

- **All 20 clients in scope** show cross-border activity — no coverage gap within 
  this dataset.
- **`intercompany`** (~half of value/volume) is internal treasury movement, not 
  contestable business — excluded from the wallet estimate.
- **`trade`** is the main usable signal. Within it, **448 rows have a populated `memo`** 
  revealing hidden lending activity (bridging facilities, syndicate participations, 
  drawdowns) — split out as an Investment Banking signal.
- **`other`** (24k rows) statistically matches `trade` on every dimension checked 
  (direction split, country spread, no memo overlap) — looks like a residual/
  unclassified bucket rather than a real category, so it's folded into `trade`.
- **`currency_pair`** is near-uniform across 5 currencies — weak signal, not used as a 
  basis for wallet sizing.

## Exploring Transactional Banking Data

Same approach as cross-border payments: check client coverage, category breakdowns, 
and whether categorisation fields (`leg_type`, `reference`, `memo`) fully account for 
the data or hide anything unexpected.

In [66]:
transactional_banking.head()

,transaction_id,entity_id,entity_name,sector,date,leg_type,direction,amount_zar,currency,channel,beneficiary_name,reference,memo
0,TXN40610803,E01,BHP Group,mining,2023-07-01,collections,inbound,63878.473869,ZAR,EFT,Continental Metals Trading House,INV-662227,NaN
1,TXN12547643,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,53220.970000,ZAR,EFT,Sunrise Cold Chain Logistics,INV-591371,NaN
2,TXN37710224,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,12309.970000,ZAR,SWIFT,Sunrise Cold Chain Logistics,PO-687736,NaN
3,TXN65618575,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,6308.300000,ZAR,Internal Transfer,Sunrise Cold Chain Logistics,INV-139304,NaN
4,TXN50222056,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,55346.750000,ZAR,Internal Transfer,Cape Wholesale Distributors,INV-556985,NaN


### Initial exploration

Checking client coverage, `leg_type` and `channel` breakdowns, currency consistency, 
memo fill rate, and activity spread across clients.

In [77]:
print(transactional_banking['entity_id'].nunique(), "clients present (matches cross-border scope)")

print("\nLeg type counts:")
print(transactional_banking['leg_type'].value_counts())

print("\nDirection split:")
print(transactional_banking['direction'].value_counts())

print("\nChannel counts:")
print(transactional_banking['channel'].value_counts())

print("\nCurrency counts (check for non-ZAR):")
print(transactional_banking['currency'].value_counts())

print("\nMemo populated rows:", transactional_banking['memo'].notna().sum(),
      "of", len(transactional_banking))

print("\nTransactions per client:")
print(transactional_banking.groupby('entity_id')['transaction_id'].count().describe())

20 clients present (matches cross-border scope)

Leg type counts:
leg_type
collections            1131589
supplier_payments       976857
intercompany_sweeps     672410
payroll                  16592
tax                       5427
Name: count, dtype: int64

Direction split:
direction
inbound     1467474
outbound    1335401
Name: count, dtype: int64

Channel counts:
channel
EFT                  1261649
RTC                   560663
Internal Transfer     419770
SWIFT                 280539
Debit Order           280254
Name: count, dtype: int64

Currency counts (check for non-ZAR):
currency
ZAR    2802875
Name: count, dtype: int64

Memo populated rows: 3657 of 2802875

Transactions per client:
count        20.000000
mean     140143.750000
std      234127.387411
min         797.000000
25%       12641.250000
50%       41363.500000
75%      144743.250000
max      933425.000000
Name: transaction_id, dtype: float64


### Finding 1 — Same 20 clients as cross-border

Client coverage matches the cross-border payments dataset exactly (confirmed formally 
in the cross-dataset check below), consistent with the data-scope note above.

### Finding 2 — Currency has a casing inconsistency

`ZAR` and `zar` appear as separate values for the same currency — needs normalising 
before any aggregation.

In [68]:
transactional_banking['currency'] = transactional_banking['currency'].str.upper()
print(transactional_banking['currency'].value_counts())  # confirm merge worked

currency
ZAR    2802875
Name: count, dtype: int64


### Does `reference` prefix map cleanly onto `leg_type`?

Checking whether the `reference` field's prefix (INV, PO, SWEEP, etc.) fully explains 
each `leg_type` category, or hides sub-categories within it.

In [69]:
print(pd.crosstab(transactional_banking['reference'].str.extract(r'^([A-Z]+\d*)-')[0],
                   transactional_banking['leg_type']))

leg_type  collections  intercompany_sweeps  payroll  supplier_payments   tax
0                                                                           
CALL                0                  406        0                  0     0
CIT                 0                    0        0                  0   551
INV           1131589                    0        0             683676     0
LOAN                0                  374        0                  0     0
MM                  0                  384        0                  0     0
PAYE                0                    0        0                  0  1658
PAYROLL             0                    0    16592                  0     0
PO                  0                    0        0             293181     0
PROV                0                    0        0                  0   793
SWEEP               0               670866        0                  0     0
TERM                0                  380        0                  0     0

### Finding 3 — `intercompany_sweeps` hides a sophisticated treasury sub-signal

`intercompany_sweeps` (672,410) doesn't map to a single reference prefix. It splits 
into SWEEP (670,866, plain internal transfers) plus CALL, LOAN, MM, and TERM 
(406+374+384+380 = 1,544) — money-market placements, term deposits, loans, and call 
accounts between related entities. These are excluded from the contestable wallet along 
with the rest of `intercompany_sweeps`, but flagged separately as a more sophisticated 
treasury relationship signal.

### Finding 4 — `tax` initially had 2,425 unmatched rows

CIT + PAYE + PROV only summed to 3,002 of the 5,427 `tax` rows. The remaining rows use 
a reference format (`VAT201-202307`) with digits in the prefix, which the original 
regex (letters only) didn't catch.

In [70]:
tax_rows = transactional_banking[transactional_banking['leg_type'] == 'tax']
unmatched = tax_rows[tax_rows['reference'].str.extract(r'^([A-Z]+)-')[0].isna()]
print(unmatched['reference'].head(10))

667     VAT201-202307
1004    VAT201-202307
1223    VAT201-202307
1473    VAT201-202307
1474    VAT201-202307
1769    VAT201-202307
1947    VAT201-202307
2796    VAT201-202307
2803    VAT201-202307
2853    VAT201-202307
Name: reference, dtype: str


Confirmed as VAT201 (VAT return) references. Widening the regex to allow trailing 
digits in the prefix resolves it — `tax` now fully accounts for CIT (551) + PAYE 
(1,658) + PROV (793) + VAT201 (2,425) = 5,427, matching exactly. Every `leg_type` 
category is now fully decomposed with no unmatched rows.

In [71]:
print(transactional_banking['reference'].str.extract(r'^([A-Z]+\d*)-')[0].value_counts())

0
INV        1815265
SWEEP       670866
PO          293181
PAYROLL      16592
VAT201        2425
PAYE          1658
PROV           793
CIT            551
CALL           406
MM             384
TERM           380
LOAN           374
Name: count, dtype: int64


### Investigating `memo` — same check as cross-border

Checking which `leg_type`s the sparse memo field clusters in, and what it says.

In [72]:
memo_rows = transactional_banking[transactional_banking['memo'].notna()]
print(memo_rows['leg_type'].value_counts())
print(memo_rows['memo'].str.split(' - ref| - ext\\. ref', regex=True).str[0].value_counts())

leg_type
supplier_payments      2113
intercompany_sweeps    1544
Name: count, dtype: int64
memo
Settlement re: facility drawdown      949
Bridging facility settlement          925
Loan drawdown proceeds                893
Syndicate participation settlement    890
Name: count, dtype: int64


### Finding 5 — Lending activity hides inside two categories, not just sweeps

Memo-tagged rows split as: `supplier_payments` (2,113) and `intercompany_sweeps` 
(1,544 — exactly matching the CALL/LOAN/MM/TERM count above, cross-validating Finding 3 
independently). The memo phrases are the same financing language seen in cross-border 
(bridging facility settlement, syndicate participation, loan drawdown proceeds).

This means some `supplier_payments` rows — which look like ordinary commercial 
activity — are actually lending/Investment Banking activity. Combined with the 448 
memo-tagged rows in cross-border, that's **2,561 transactions across all datasets so 
far** that a simple category split would miscount as routine trade or transactional 
activity.

### Cross-dataset checks

Confirming whether cross-border and transactional show the same 20 active clients, and 
whether any transactions are double-counted across the two datasets.

In [73]:
cb_clients = set(cross_border_payments['entity_id'].unique())
tb_clients = set(transactional_banking['entity_id'].unique())
print("Same 20 clients in both:", cb_clients == tb_clients)

print("Shared transaction_ids:", cross_border_payments['transaction_id'].isin(transactional_banking['transaction_id']).sum())

Same 20 clients in both: True
Shared transaction_ids: 0


### Transactional Banking — Key Findings

- **Same 20 clients** as cross-border payments — confirms consistent scope across both 
  datasets.
- **No shared `transaction_id`s** with cross-border payments — safe to combine totals 
  without double-counting.
- **`intercompany_sweeps`** (672,410) is mostly internal treasury movement, excluded 
  from the contestable wallet — except 1,544 rows that are money-market/loan/term 
  activity between related entities, flagged as a separate treasury signal.
- **2,561 transactions across both datasets** (2,113 here + 448 in cross-border) are 
  lending/financing activity hiding inside `supplier_payments` and `trade` categories, 
  identifiable via the `memo` field — treated as a distinct Investment Banking signal.
- **`tax`** fully decomposes into CIT, PAYE, PROV, and VAT201 — no unexplained rows.
- **`currency`** normalised (ZAR/zar casing merged) — single currency confirmed.

## Exploring Trade Finance Data

Same approach as the previous two datasets: check client coverage, category 
breakdowns, and whether categorisation fields fully account for the data or hide 
anything unexpected — this time for letters of credit, guarantees, and export 
collections.

In [74]:
trade_finance.head()

,instrument_id,entity_id,entity_name,sector,date,instrument_type,direction,tenor_days,value_zar,counterparty_country,commodity_or_contract_type,status,beneficiary_name,reference,memo
0,TF67401938,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,60,13499920.55,United Arab Emirates,agri_produce,issued,Silverline Trading Co.,LC-471415,NaN
1,TF91455580,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,365,558304.97,Switzerland,iron_ore,settled,Global Commodities Marketing,LC-266865,NaN
2,TF31370953,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,30,4042335.97,United Kingdom,agri_produce,settled,Pacific International Trading House,LC-946978,NaN
3,TF13634137,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,import,365,171042.89,China,platinum_group_metals,active,Silverline Resources Trading,LC-553503,NaN
4,TF86438695,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,120,331531.78,Netherlands,electronics,settled,Silverline Resources Trading,LC-260628,NaN


### Initial exploration

Checking client coverage, `instrument_type` and `status` breakdowns, direction split, 
tenor distribution, and memo fill rate — same pattern as the previous two datasets.

In [78]:
print(trade_finance['entity_id'].nunique(), "clients present (matches scope)")

print("\nInstrument type counts:")
print(trade_finance['instrument_type'].value_counts())

print("\nStatus counts:")
print(trade_finance['status'].value_counts())

print("\nDirection split:")
print(trade_finance['direction'].value_counts())

print("\nTenor (days) distribution:")
print(trade_finance['tenor_days'].describe())

print("\nCommodity/contract type counts:")
print(trade_finance['commodity_or_contract_type'].value_counts())

print("\nMemo populated rows:", trade_finance['memo'].notna().sum(),
      "of", len(trade_finance))

print("\nTransactions per client:")
print(trade_finance.groupby('entity_id')['instrument_id'].count().describe())

print("\nReference prefix counts:")
print(trade_finance['reference'].str.extract(r'^([A-Z]+\d*)-')[0].value_counts())

20 clients present (matches scope)

Instrument type counts:
instrument_type
letters_of_credit     8134
export_collections    6793
guarantees            5376
Name: count, dtype: int64

Status counts:
status
settled    8632
active     7066
issued     2995
expired    1610
Name: count, dtype: int64

Direction split:
direction
import    11294
export     9009
Name: count, dtype: int64

Tenor (days) distribution:
count    20303.000000
mean       107.866325
std         81.376315
min         30.000000
25%         60.000000
50%         90.000000
75%        120.000000
max        365.000000
Name: tenor_days, dtype: float64

Commodity/contract type counts:
commodity_or_contract_type
bid_bond                     1823
advance_payment_guarantee    1794
performance_guarantee        1759
platinum_group_metals        1291
electronics                  1290
coal                         1277
copper                       1273
telecom_equipment            1254
pharmaceuticals              1242
iron_ore       

### Finding 1 — Same 20 clients again, third dataset in a row

Matches cross-border and transactional exactly, consistent with the data-scope note 
above.

### Finding 2 — Reference prefix maps cleanly onto `instrument_type`

Unlike the previous two datasets, this one has no mismatch: LC (8,134), COLL (6,793), 
GTE (5,376) map exactly onto letters_of_credit, export_collections, and guarantees. No 
hidden sub-categories via reference — this field is clean.

### Investigating `commodity_or_contract_type` — does it hold two different things?

The value list mixes what look like guarantee sub-types (bid_bond, advance_payment_
guarantee, performance_guarantee) with commodity names (iron_ore, gold, electronics, 
etc). The three guarantee-type values sum to 1,823+1,794+1,759 = 5,376 — exactly 
matching the `guarantees` instrument count. Checking whether this field serves a dual 
purpose depending on `instrument_type`.

In [79]:
print(pd.crosstab(trade_finance['commodity_or_contract_type'], trade_finance['instrument_type']))

instrument_type             export_collections  guarantees  letters_of_credit
commodity_or_contract_type                                                   
advance_payment_guarantee                    0        1794                  0
agri_produce                               564           0                641
bid_bond                                     0        1823                  0
chemicals                                  565           0                675
coal                                       570           0                707
consumer_goods                             528           0                672
copper                                     562           0                711
electronics                                559           0                731
gold                                       586           0                648
iron_ore                                   576           0                664
manufactured_goods                         544           0      


### Finding 3 — `commodity_or_contract_type` is dual-purpose

Confirmed via crosstab: the field holds guarantee sub-types (bid_bond, advance_payment_
guarantee, performance_guarantee) exclusively under `instrument_type == guarantees`, 
and commodity/goods categories exclusively under `letters_of_credit`/
`export_collections` — zero overlap between the two groups. This field should be split 
into two separate derived columns (e.g. `guarantee_subtype` and `commodity_type`) 
before use in aggregation, rather than treated as one flat category.

In [80]:
memo_rows = trade_finance[trade_finance['memo'].notna()]
print(memo_rows['instrument_type'].value_counts())
print(memo_rows['memo'].str.split(' - ref| - ext\\. ref', regex=True).str[0].value_counts())

instrument_type
letters_of_credit     46
export_collections    31
guarantees            17
Name: count, dtype: int64
memo
Settlement re: facility drawdown      25
Bridging facility settlement          25
Loan drawdown proceeds                24
Syndicate participation settlement    20
Name: count, dtype: int64


### Finding 4 — Trade finance has its own small lending signal via memo

94 memo-populated rows (46 letters_of_credit, 31 export_collections, 17 guarantees), 
same four financing phrases as the other two datasets (bridging facility settlement, 
syndicate participation, loan drawdown proceeds, facility drawdown settlement). 
Consistent with the pattern found across all three datasets: memo flags transactions 
that are actually lending/Investment Banking activity, regardless of which product 
category they're nominally filed under.

### Cross-dataset checks

Same 20-client scope as the other two datasets, confirmed. No shared IDs with 
transactional banking — safe to combine without double-counting.

In [81]:
tf_clients = set(trade_finance['entity_id'].unique())
tb_clients = set(transactional_banking['entity_id'].unique())
print("Same client set as transactional:", tf_clients == tb_clients)
print("Shared instrument/transaction IDs with transactional:",
      trade_finance['instrument_id'].isin(transactional_banking['transaction_id']).sum())

Same client set as transactional: True
Shared instrument/transaction IDs with transactional: 0


### Trade Finance — Key Findings

- **Same 20-client scope** as the other two datasets, confirmed.
- **`commodity_or_contract_type` is dual-purpose** — guarantee sub-types under 
  `guarantees`, commodity categories under `letters_of_credit`/`export_collections`, 
  zero overlap. Should be split into two derived columns before use.
- **`reference` prefix maps cleanly** onto `instrument_type` (LC/COLL/GTE) — no hidden 
  sub-categories via reference, unlike transactional banking.
- **94 memo-tagged rows** are lending/Investment Banking activity hiding inside trade 
  instruments — same four financing phrases as the other two datasets.
- **No shared IDs** with transactional banking — safe to combine across all three 
  datasets without double-counting.